# 1. Setup & Environment
- 공간 데이터 및 시계열 전이(Transition) 연산용 라이브러리 로드
- 판다스 디스플레이 포맷 및 경고 필터링 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

# 2. Configuration & Transition Matrix Definitions
- 집계구 경계 및 핫스팟 비교 산출물 입출력 디렉터리 설정
- 서울시 25개 자치구 코드 매핑 및 2개 모형(2SFCA, Gravity) 파라미터 정의

In [2]:
# 2. 경로 및 분석 파라미터 정의
BASE_DIR = Path("/mnt/cowork/EV")
BOUNDARY_FP = BASE_DIR / "input/raw/집계구_2016/집계구.shp"
DIR_OUTPUT = BASE_DIR / "output"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# 입력 및 최종 산출물 경로 (_mw)
FP_HOTSPOT_MW = DIR_OUTPUT / "two_model_hotspot_k30_mw.csv"
FP_HOTSPOT_ORIG = DIR_OUTPUT / "two_model_hotspot_k30.csv"
OUT_FP_MW = DIR_OUTPUT / "hotspot_transition_2021_2024_mw.csv"

# 분석 대상 모형 및 비교 시점
MODELS = ["2SFCA", "Gravity"]
BASE_YEAR, TARGET_YEAR = 2021, 2024

# 서울시 시군구 코드 매핑 테이블
GU_MAP = {
    "11010": "종로구", "11020": "중구", "11030": "용산구", "11040": "성동구", "11050": "광진구",
    "11060": "동대문구", "11070": "중랑구", "11080": "성북구", "11090": "강북구", "11100": "도봉구",
    "11110": "노원구", "11120": "은평구", "11130": "서대문구", "11140": "마포구", "11150": "양천구",
    "11160": "강서구", "11170": "구로구", "11180": "금천구", "11190": "영등포구", "11200": "동작구",
    "11210": "관악구", "11220": "서초구", "11230": "강남구", "11240": "송파구", "11250": "강동구",
}

print(f">> 분석 시계열: {BASE_YEAR}년 -> {TARGET_YEAR}년 전이 분석")
print(f">> 비교 대상 모형: {MODELS}")
print(f">> 최종 산출물 경로: {OUT_FP_MW}")

>> 분석 시계열: 2021년 -> 2024년 전이 분석
>> 비교 대상 모형: ['2SFCA', 'Gravity']
>> 최종 산출물 경로: /mnt/cowork/EV/output/hotspot_transition_2021_2024_mw.csv


# 3. Vectorized Transition Engine
- 행 단위 반복문(apply)을 대체하는 NumPy 고속 벡터화 군집 전이 분류 함수 정의
- 지속콜드, 신규악화, 개선, 지속핫, 신규콜드, 콜드탈출 등 6대 핵심 전이 유형화

In [3]:
# 3. 고속 벡터화 전이 분류 함수 정의
def classify_transition_vectorized(s_start: pd.Series, s_end: pd.Series) -> np.ndarray:
    """2021년과 2024년 군집 판정 시리즈를 입력받아 전이 유형을 벡터 연산으로 분류"""
    # 1. 3대 단순 범주(Hot, Cold, NotSig)로 정규화
    clean_start = s_start.replace({"Hot Spot": "Hot", "Cold Spot": "Cold", "Not Sig": "NotSig"}).fillna("NotSig").values
    clean_end = s_end.replace({"Hot Spot": "Hot", "Cold Spot": "Cold", "Not Sig": "NotSig"}).fillna("NotSig").values

    # 2. 우선순위 전이 조건 벡터 정의
    cond_persistent_cold = (clean_start == "Cold") & (clean_end == "Cold")
    cond_worsening       = (clean_start == "Hot") & (clean_end == "Cold")
    cond_improving       = (clean_start == "Cold") & (clean_end == "Hot")
    cond_persistent_hot  = (clean_start == "Hot") & (clean_end == "Hot")
    cond_new_cold        = (clean_start == "NotSig") & (clean_end == "Cold")
    cond_escaped_cold    = (clean_start == "Cold") & (clean_end == "NotSig")

    conditions = [
        cond_persistent_cold,
        cond_worsening,
        cond_improving,
        cond_persistent_hot,
        cond_new_cold,
        cond_escaped_cold,
    ]

    choices = [
        "지속콜드",
        "신규악화(Hot->Cold)",
        "개선(Cold->Hot)",
        "지속핫",
        "신규콜드(NotSig->Cold)",
        "콜드탈출(Cold->NotSig)",
    ]

    # 기본값: '시작->종료' 포맷
    fallback = np.char.add(np.char.add(clean_start.astype(str), "->"), clean_end.astype(str))
    
    return np.select(conditions, choices, default=fallback)

# 4. Data Loaders & Spatial Attribute Mapper
- 서울시 14,979개 집계구 폴리곤 및 행정동/자치구 속성 테이블 로드
- 모형별 핫스팟 결과 CSV(`two_model_hotspot_k30_mw.csv`) 로드

In [4]:
# 4. 집계구 경계 및 핫스팟 데이터 로드
gdf_boundary = gpd.read_file(BOUNDARY_FP)
gdf_boundary["TOT_REG_CD"] = gdf_boundary["TOT_REG_CD"].astype(str)

# 서울시 집계구 필터링 및 행정동 메타데이터 테이블 구성
df_oa_meta = (
    gdf_boundary[gdf_boundary["TOT_REG_CD"].str.startswith("11")][["TOT_REG_CD", "ADM_NM"]]
    .rename(columns={"TOT_REG_CD": "oa_code"})
    .reset_index(drop=True)
)
df_oa_meta["gu"] = df_oa_meta["oa_code"].str[:5].map(GU_MAP)
print(f">> 서울시 집계구 속성 로드 완료: 총 {len(df_oa_meta):,}개 집계구")

# 핫스팟 데이터 로드 (_mw 우선)
if FP_HOTSPOT_MW.exists():
    fp_in = FP_HOTSPOT_MW
    print(f">> 신규 리팩토링 산출물 로드: {fp_in.name}")
elif FP_HOTSPOT_ORIG.exists():
    fp_in = FP_HOTSPOT_ORIG
    print(f">> 기존 원본 산출물 로드: {fp_in.name}")
else:
    raise FileNotFoundError("입력 핫스팟 CSV 파일을 찾을 수 없습니다.")

df_hotspot = pd.read_csv(fp_in, dtype={"oa_code": str})

>> 서울시 집계구 속성 로드 완료: 총 19,153개 집계구
>> 신규 리팩토링 산출물 로드: two_model_hotspot_k30_mw.csv


# 5. Batch Transition Pipeline & CSV Export
- 2SFCA 및 Gravity 모형별 2021-2024 피벗 테이블 생성 및 전이 분류 실행
- 공간 메타데이터(구, 동) 결합 후 최종 CSV(`hotspot_transition_2021_2024_mw.csv`) 저장

In [5]:
# 5. 배치 전이 분석 및 결과 내보내기
print("=" * 80)
print("RUNNING: SPATIOTEMPORAL TRANSITION ANALYSIS (2021 vs 2024)")
print("=" * 80)

transition_dfs = []

for model in MODELS:
    # 당해 모형 및 2개년 데이터 추출
    sub = df_hotspot[(df_hotspot["model"] == model) & (df_hotspot["year"].isin([BASE_YEAR, TARGET_YEAR]))]
    
    # Year 기준 가로 피벗
    piv = sub.pivot(index="oa_code", columns="year", values="gi_class").reset_index()
    piv.columns = [str(c) for c in piv.columns]
    
    # 벡터화 전이 유형 산출
    piv["transition"] = classify_transition_vectorized(piv[str(BASE_YEAR)], piv[str(TARGET_YEAR)])
    piv["model"] = model
    transition_dfs.append(piv)
    
    print(f"\n[{model}] 전이 유형별 집계구 분포:")
    print(piv["transition"].value_counts().to_string())

# 전체 모형 병합 및 행정동 속성 조인
df_transition = pd.concat(transition_dfs, ignore_index=True)
df_transition = df_transition.merge(df_oa_meta, on="oa_code", how="left")

# 최종 CSV 내보내기 (_mw)
df_transition.to_csv(OUT_FP_MW, index=False, encoding="utf-8-sig")
print(f"\n>> 전이 분석 산출물 저장 완료: {OUT_FP_MW} (총 {len(df_transition):,}행)")

RUNNING: SPATIOTEMPORAL TRANSITION ANALYSIS (2021 vs 2024)

[2SFCA] 전이 유형별 집계구 분포:
transition
지속콜드                  5386
지속핫                   3883
신규콜드(NotSig->Cold)    2199
NotSig->NotSig        1974
콜드탈출(Cold->NotSig)    1361
신규악화(Hot->Cold)       1278
Hot->NotSig           1227
NotSig->Hot           1111
개선(Cold->Hot)          734

[Gravity] 전이 유형별 집계구 분포:
transition
지속콜드                  5749
지속핫                   4521
NotSig->NotSig        2201
신규콜드(NotSig->Cold)    1836
NotSig->Hot           1751
콜드탈출(Cold->NotSig)    1234
Hot->NotSig           1072
개선(Cold->Hot)          624
신규악화(Hot->Cold)        165

>> 전이 분석 산출물 저장 완료: /mnt/cowork/EV/output/hotspot_transition_2021_2024_mw.csv (총 38,306행)


# 6. Policy Priority Areas Ranking (Persistent Cold & Sudden Degradation)
- 모형별 '지속콜드(구조적 취약)' 및 '신규악화(Hot->Cold, 급격한 취약화)' 상위 행정동 집계

In [6]:
# 6. 취약 유형별 상위 행정동 랭킹 도출
print("=" * 80)
print("             모형별 핵심 취약 전이 지역 상위 10개 행정동 요약")
print("=" * 80)

for model in MODELS:
    print(f"\n=== [{model}] 1. 지속콜드 (2021 Cold -> 2024 Cold) 최상위 행정동 ===")
    df_pc = df_transition[(df_transition["model"] == model) & (df_transition["transition"] == "지속콜드")]
    pc_ranking = (
        df_pc.groupby(["gu", "ADM_NM"])
        .size()
        .reset_index(name="persistent_cold_oa_count")
        .sort_values(by="persistent_cold_oa_count", ascending=False)
        .head(10)
    )
    display(pc_ranking.reset_index(drop=True))

    print(f"\n=== [{model}] 2. 신규악화 (2021 Hot -> 2024 Cold) 최상위 행정동 ===")
    df_worse = df_transition[(df_transition["model"] == model) & (df_transition["transition"] == "신규악화(Hot->Cold)")]
    worse_ranking = (
        df_worse.groupby(["gu", "ADM_NM"])
        .size()
        .reset_index(name="worsened_oa_count")
        .sort_values(by="worsened_oa_count", ascending=False)
        .head(10)
    )
    display(worse_ranking.reset_index(drop=True))

             모형별 핵심 취약 전이 지역 상위 10개 행정동 요약

=== [2SFCA] 1. 지속콜드 (2021 Cold -> 2024 Cold) 최상위 행정동 ===


,gu,ADM_NM,persistent_cold_oa_count
0,강동구,길동,95
1,동작구,상도1동,88
2,강남구,역삼2동,72
3,강동구,암사1동,71
4,동작구,흑석동,69
5,강동구,강일동,69
6,송파구,잠실2동,68
7,관악구,은천동,67
8,송파구,잠실3동,67
9,관악구,성현동,66



=== [2SFCA] 2. 신규악화 (2021 Hot -> 2024 Cold) 최상위 행정동 ===


,gu,ADM_NM,worsened_oa_count
0,마포구,공덕동,77
1,마포구,아현동,53
2,마포구,도화동,42
3,동대문구,제기동,41
4,서대문구,충현동,38
5,은평구,불광2동,37
6,서대문구,천연동,34
7,용산구,청파동,34
8,성동구,응봉동,33
9,서초구,서초3동,32



=== [Gravity] 1. 지속콜드 (2021 Cold -> 2024 Cold) 최상위 행정동 ===


,gu,ADM_NM,persistent_cold_oa_count
0,은평구,진관동,109
1,강동구,길동,95
2,은평구,역촌동,93
3,구로구,오류2동,83
4,동작구,상도1동,82
5,은평구,불광1동,80
6,강동구,강일동,69
7,금천구,시흥1동,68
8,관악구,은천동,67
9,관악구,성현동,66



=== [Gravity] 2. 신규악화 (2021 Hot -> 2024 Cold) 최상위 행정동 ===


,gu,ADM_NM,worsened_oa_count
0,서초구,내곡동,27
1,성동구,옥수동,20
2,서초구,반포본동,17
3,동대문구,제기동,17
4,서대문구,연희동,14
5,서대문구,신촌동,9
6,서대문구,홍제3동,8
7,서대문구,홍제2동,8
8,도봉구,방학3동,6
9,서초구,잠원동,5


# 7. Cross-Model Mechanism Diagnosis
- 2SFCA와 Gravity 간 교차 분석을 통한 공간 취약 원인 규명
  * 교차 지속콜드: 인구 수요와 무관하게 공급 자체가 절대적으로 결여된 **물리적·구조적 결핍지**
  * 2SFCA 전용 지속콜드: 공급은 존재하나 생활인구 과밀로 인해 병목이 발생한 **수요 압박형 결핍지**

In [7]:
# 7. 모형 간 메커니즘 교차진단 연산
set_2sfca_cold = set(df_transition[(df_transition["model"] == "2SFCA") & (df_transition["transition"] == "지속콜드")]["oa_code"])
set_gravity_cold = set(df_transition[(df_transition["model"] == "Gravity") & (df_transition["transition"] == "지속콜드")]["oa_code"])

both_cold = set_2sfca_cold & set_gravity_cold
only_2sfca = set_2sfca_cold - set_gravity_cold
only_gravity = set_gravity_cold - set_2sfca_cold

print("=" * 80)
print("             모형 교차진단: EV 충전 인프라 결핍 메커니즘 분석")
print("=" * 80)
print(f"- 2SFCA 기준 지속콜드 집계구 수  : {len(set_2sfca_cold):,}개")
print(f"- Gravity 기준 지속콜드 집계구 수: {len(set_gravity_cold):,}개")
print("-" * 80)
print(f"1. 구조적 절대 결핍 지역 (두 모형 모두 지속콜드)   : {len(both_cold):,}개")
print(f"   -> 공급 인프라 자체가 물리적으로 부족하여 우선 확충이 필요한 절대 소외지")
print(f"2. 수요 압박형 결핍 지역 (2SFCA만 지속콜드, Gravity 정상): {len(only_2sfca):,}개")
print(f"   -> 충전소는 도달 범위 내에 있으나 인구 과밀로 수요 경쟁이 심화된 병목지")
print(f"3. 단순 물리적 결핍 지역 (Gravity만 지속콜드, 2SFCA 정상): {len(only_gravity):,}개")
print("=" * 80)

             모형 교차진단: EV 충전 인프라 결핍 메커니즘 분석
- 2SFCA 기준 지속콜드 집계구 수  : 5,386개
- Gravity 기준 지속콜드 집계구 수: 5,749개
--------------------------------------------------------------------------------
1. 구조적 절대 결핍 지역 (두 모형 모두 지속콜드)   : 3,335개
   -> 공급 인프라 자체가 물리적으로 부족하여 우선 확충이 필요한 절대 소외지
2. 수요 압박형 결핍 지역 (2SFCA만 지속콜드, Gravity 정상): 2,051개
   -> 충전소는 도달 범위 내에 있으나 인구 과밀로 수요 경쟁이 심화된 병목지
3. 단순 물리적 결핍 지역 (Gravity만 지속콜드, 2SFCA 정상): 2,414개


# 8. Result Validation (Comparison with Original Outputs)
- 기존 원본 산출물(`hotspot_transition_2021_2024.csv`)과 신규 산출물(`_mw.csv`) 간 전이 분류 일치율 전수 검증

In [8]:
# 8. 원본 산출물 vs 신규 산출물(_mw) 정밀 일치율 검증
fp_orig_trans = DIR_OUTPUT / "hotspot_transition_2021_2024.csv"
fp_new_trans = OUT_FP_MW

if not fp_orig_trans.exists():
    print(f"[!] 비교할 원본 결과 파일이 존재하지 않습니다: {fp_orig_trans}")
elif not fp_new_trans.exists():
    print(f"[!] 신규 산출물 파일이 생성되지 않았습니다: {fp_new_trans}")
else:
    df_orig = pd.read_csv(fp_orig_trans, dtype={"oa_code": str})
    df_new = pd.read_csv(fp_new_trans, dtype={"oa_code": str})

    comp = df_orig.merge(df_new, on=["model", "oa_code"], suffixes=("_orig", "_new"))
    comp["is_match"] = comp["transition_orig"] == comp["transition_new"]

    total_count = len(comp)
    match_count = comp["is_match"].sum()
    match_rate = (match_count / total_count) * 100

    print("=" * 80)
    print("           전이 분석 산출물 원본 vs 리팩토링 코드 수치 검증 요약표")
    print("=" * 80)
    print(f"- 총 검증 레코드 수  : {total_count:,}개")
    print(f"- 전이 분류 일치 건수: {match_count:,}개")
    print(f"- 전체 일치율 (Match Rate): {match_rate:.4f}%")

    # 모형별 상세 일치율
    piv_val = comp.groupby("model")["is_match"].agg(
        총레코드="count",
        일치건수="sum",
        일치율=lambda x: f"{(x.mean()*100):.2f}%"
    )
    display(piv_val)

    if match_count == total_count:
        print(">> [판정] 2SFCA 및 Gravity 모형의 모든 시계열 전이 분류가 기존 원본 산출물과 100% 완벽히 일치합니다.")
    else:
        print(">> [판정] 불일치 레코드가 확인되었습니다. 아래 샘플을 점검해주세요.")
        display(comp[~comp["is_match"]].head(5))

           전이 분석 산출물 원본 vs 리팩토링 코드 수치 검증 요약표
- 총 검증 레코드 수  : 38,306개
- 전이 분류 일치 건수: 38,306개
- 전체 일치율 (Match Rate): 100.0000%


,총레코드,일치건수,일치율
model,,,
2SFCA,19153,19153,100.00%
Gravity,19153,19153,100.00%


>> [판정] 2SFCA 및 Gravity 모형의 모든 시계열 전이 분류가 기존 원본 산출물과 100% 완벽히 일치합니다.
